# [실습] Model Context Protocol

Agent의 Tool Calling을 표준화한 프로토콜인 Model Context Protocol(MCP)에 대해 알아보겠습니다.   

MCP 서버를 직접 구축하고, LangChain 에이전트와 연결합니다.   


## 학습 목표

- MCP의 구성 요소와 Transport 방식에 대해 이해합니다.
- FastMCP로 MCP 서버를 만들고 LangChain Agent에 연결합니다.
- MCP가 내부적으로 주고받는 메시지를 직접 확인합니다.
- vLLM/Ollama 등의 오픈 모델을 연결하여 LLM Agent를 구성합니다.


## 환경 설정

In [1]:
%pip install langchain langchain-openai langchain-ollama fastmcp langchain-mcp-adapters slack-sdk httpx -q

Note: you may need to restart the kernel to use updated packages.


In [2]:
from dotenv import load_dotenv
import os

load_dotenv('.env', override=True)

if os.environ.get('OPENAI_API_KEY'):
    print('OpenAI API 키 설정 완료!')

OpenAI API 키 설정 완료!


---
## 플랫폼 선택

이번 실습부터는, OpenAI API 이외의 모델을 연결할 수 있습니다.


vLLM/Ollama 등의 서빙을 통해, 오픈 모델을 연결하는 방법에 대해서는 별도로 전달된 가이드 문서를 참고해 주세요.

In [3]:
from select_model import load_model

# openai: 상용 API (기본). vllm, ollama: 오픈 모델 서버.
PLATFORM = 'openai'        # 'openai' | 'vllm' | 'ollama'

# 플랫폼별 모델명. vllm은 None이면 서버가 올린 첫 번째 모델을 자동으로 연결합니다.
MODEL_NAME = {
    'openai': 'gpt-5.2',
    'vllm': None,
    'ollama': 'qwen3.6:latest',
}[PLATFORM]

# 모델별 추가 인자를 전달할 수 있습니다.
MODEL_KWARGS = {
    'openai': {'reasoning_effort': 'low'},
    'vllm': {},
    'ollama': {},
}[PLATFORM]

model = load_model(MODEL_NAME, platform=PLATFORM, **MODEL_KWARGS)
print(f'{PLATFORM} 모델 로드:', type(model).__name__)

openai 모델 로드: ChatOpenAI


## 1. MCP(Model Context Protocol)

MCP는 LLM에 도구와 데이터를 제공하기 위한 표준 통신 규약입니다.   

다양한 SDK/ADK마다 다른 툴 콜링 메커니즘을 통합하기 위해, Tool과 LLM Agent 관계를   
Server/Client 형식으로 연동하고, STDIO/Transport 형식의 통신 방법을 제안합니다.   

MCP는 크로스 플랫폼을 지향하며, Tool에서는 포함하기 어려운 인증/미들웨어 등의 기능을 연동할 수 있습니다.  


### MCP의 주요 구성 요소
- Tool: 모델이 호출하는 함수로, LangChain의 Tool과 동일합니다. (`tools/call`)
- Resource: 클라이언트가 읽는 읽기 전용 데이터를 전달합니다. (`resources/read`)
- Prompt: 서버 레벨에서 프롬프트 템플릿을 정의합니다. (`prompts/get`)
- Transport: 서버와 클라이언트가 통신하는 방식입니다. (stdio, Streamable-HTTP)

### MCP의 Transport 형식
MCP 서버와 클라이언트의 통신은 두 가지로 구분됩니다.
- stdio: 클라이언트가 CLI 인터페이스를 통해 서버를 커맨드라인으로 실행합니다.
  python, uvx, npx 등의 커맨드가 포함되어야 합니다.   
  서버는 클라이언트가 요청할 때 켜지며, 결과를 반환합니다.

- streamable-http: 서버는 HTTP 통신을 위해 특정 주소와 포트에서 실행됩니다.   
  이후 해당 주소에 클라이언트가 요청을 보내면 서버가 처리합니다.


### MCP를 사용하는 이유

1. MCP를 통해, 도구를 별도 서버로 분리하여 여러 에이전트가 같은 서버를 사용하도록 구성할 수 있습니다.
2. Cursor, Claude Desktop, LangChain와 같이 서로 다른 프레임워크의 클라이언트에서 호환됩니다.
3. 도구 코드와 토큰이 Server-side에 존재하므로, 에이전트 코드와 분리되는 이점을 갖습니다.

### MCP가 필요하지 않은 경우
다음 경우에는 일반 `@tool` 데코레이터가 더 간단합니다.

- 도구를 내부 환경에서만 쓰는 경우
- 네트워크를 통해 다른 클라이언트와 공유하지 않는 경우


---
## 2. FastMCP로 MCP 서버 구축

FastMCP는 Python으로 MCP 서버를 쉽게 만들 수 있는 라이브러리입니다.

streamable-http 방식의 서버를 구현해 보겠습니다.

In [5]:
%%writefile mcp_demo_server.py
from fastmcp import FastMCP

# 1. MCP 서버 인스턴스 생성
mcp = FastMCP(
    name="demo",
    instructions="간단한 계산과 인사말을 제공하는 MCP 서버입니다.",
)

# 2. Resource 정의 (클라이언트가 URI로 접근하는 읽기 전용 데이터)
@mcp.resource("info://about")
def get_about() -> str:
    """서버 정보를 반환합니다."""
    return "데모 MCP 서버입니다."

# 2-1. 고정값 Resource: 이 서버의 계산 정책
@mcp.resource("config://limits")
def get_limits() -> dict:
    """이 서버의 계산 정책(지원 연산, 입력 한도)을 반환합니다."""
    return {
        "supported_ops": ["add", "multiply"],
        "max_operand": 10_000_000,
        "note": "정수 연산만 지원합니다.",
    }

# 2-2. Resource Template: URI에 인자를 전달하는 동적 Resource
@mcp.resource("const://{name}")
def get_constant(name: str) -> str:
    """이름으로 상수 값을 조회합니다. (pi, e, golden)"""
    table = {
        "pi": "3.141592653589793",
        "e": "2.718281828459045",
        "golden": "1.618033988749895",
    }
    return table.get(name, f"알 수 없는 상수: {name}")

# 3. Tool 정의 (모델이 호출하는 함수)
@mcp.tool()
def add(a: int, b: int) -> str:
    """두 정수를 더합니다."""
    return f"{a} + {b} = {a + b}"

@mcp.tool()
def multiply(a: int, b: int) -> str:
    """두 정수를 곱합니다."""
    return f"{a} x {b} = {a * b}"

@mcp.tool()
def get_greeting(name: str, style: str = "formal") -> str:
    """인사말을 생성합니다."""
    if style == "casual":
        return f"안녕 {name}! 반가워~"
    return f"안녕하세요, {name}님."

# 4. Prompt 정의 (서버가 배포하는 프롬프트 템플릿)
@mcp.prompt()
def explain_calculation(expression: str) -> str:
    """주어진 수식을 단계별로 풀이해 설명하도록 지시하는 프롬프트."""
    return (
        f"다음 수식을 초등학생도 이해할 수 있게 한 단계씩 풀이해 주세요.\n\n"
        f"수식: {expression}\n\n"
        f"각 단계에서 무엇을 왜 계산하는지 한 줄로 설명하고, 마지막 줄에 최종 답을 적어 주세요."
    )

# 5. 서버 실행
if __name__ == "__main__":
    mcp.run(transport="streamable-http", port=8090)


Overwriting mcp_demo_server.py


streamable http 서버는 8090 포트를 통해 연결됩니다.

### Streamable-HTTP MCP 서버 실행하기

MCP 서버를 별도 터미널에서 실행합니다.

> 터미널 열기: VS Code에서 `Ctrl+J` 또는 별도 cmd/터미널 창
>
> ```bash
> python mcp_demo_server.py
> ```
>
> 서버가 시작되면 `http://localhost:8090/mcp`에서 접속 가능합니다.
> 종료는 해당 터미널에서 `Ctrl+C`로 합니다.

서버가 실행 중인 상태에서 아래 셀을 실행하세요.

---
## 3. MCP 클라이언트로 도구 로드

LangChain은 `langchain-mcp-adapters`의 `MultiServerMCPClient`를 통해 MCP 서버를 연결합니다.


서버의 상태를 먼저 확인해 보겠습니다.

In [7]:
# 서버 헬스체크: 응답이 오는지 먼저 확인합니다.
import httpx

try:
    r = httpx.post(
        'http://localhost:8090/mcp',
        headers={
            'Content-Type': 'application/json',
            'Accept': 'application/json, text/event-stream',
        },
        json={
            'jsonrpc': '2.0', 'id': 0, 'method': 'initialize',
            'params': {
                'protocolVersion': '2025-06-18',
                'capabilities': {},
                'clientInfo': {'name': 'healthcheck', 'version': '0'},
            },
        },
        timeout=3.0,
    )
    print('서버 응답 OK:', r.status_code)
except httpx.ConnectError:
    print('서버에 연결할 수 없습니다. 다른 터미널에서 `python mcp_demo_server.py`를 먼저 실행하세요.')

서버 응답 OK: 200


### MCP 메시지 흐름

MCP는 JSON-RPC 라는 포맷으로 메시지를 주고받습니다.

1. `initialize`: 클라이언트가 자기 소개와 protocol version을 보냅니다.
2. `notifications/initialized`: 협상 완료를 알리고 응답은 받지 않습니다.
3. `tools/list`: 서버가 가진 도구 목록을 받습니다.
4. `tools/call`: 도구를 실제로 호출합니다.

In [8]:
import json, re, httpx

URL = 'http://localhost:8090/mcp'
HEADERS_BASE = {
    'Content-Type': 'application/json',
    'Accept': 'application/json, text/event-stream',
}


def _parse_sse(text: str) -> dict:
    """`event: message\\ndata: {...}` 형식에서 첫 data 블록을 파싱합니다."""
    m = re.search(r'^data:\s*(\{.*\})\s*$', text, re.MULTILINE)
    return json.loads(m.group(1)) if m else json.loads(text)


# 1) initialize
with httpx.Client(timeout=10.0) as cli:
    r = cli.post(URL, headers=HEADERS_BASE, json={
        'jsonrpc': '2.0', 'id': 1, 'method': 'initialize',
        'params': {
            'protocolVersion': '2025-06-18',
            'capabilities': {},
            'clientInfo': {'name': 'raw-trace', 'version': '0.1'},
        },
    })
    init_resp = _parse_sse(r.text)
    session_id = r.headers.get('mcp-session-id')

print('1) initialize 응답')
print(json.dumps(init_resp['result'], ensure_ascii=False, indent=2)[:400], '...')
print(f'   (협상된 세션 ID: {session_id})')

# 2) notifications/initialized
session_headers = {**HEADERS_BASE, 'mcp-session-id': session_id} if session_id else HEADERS_BASE
with httpx.Client(timeout=10.0) as cli:
    cli.post(URL, headers=session_headers, json={
        'jsonrpc': '2.0', 'method': 'notifications/initialized', 'params': {},
    })
print('\n2) notifications/initialized 전송 완료')

# 3) tools/list
with httpx.Client(timeout=10.0) as cli:
    r = cli.post(URL, headers=session_headers, json={
        'jsonrpc': '2.0', 'id': 2, 'method': 'tools/list', 'params': {},
    })
    tools_resp = _parse_sse(r.text)

print('\n3) tools/list 응답')
for t in tools_resp['result']['tools']:
    print(f"   - {t['name']}: {t.get('description','')[:50]}")

# 4) tools/call
with httpx.Client(timeout=10.0) as cli:
    r = cli.post(URL, headers=session_headers, json={
        'jsonrpc': '2.0', 'id': 3, 'method': 'tools/call',
        'params': {'name': 'add', 'arguments': {'a': 3, 'b': 7}},
    })
    call_resp = _parse_sse(r.text)

print('\n4) tools/call 응답')
print(json.dumps(call_resp['result'], ensure_ascii=False, indent=2))

1) initialize 응답
{
  "protocolVersion": "2025-06-18",
  "capabilities": {
    "experimental": {},
    "logging": {},
    "prompts": {
      "listChanged": true
    },
    "resources": {
      "subscribe": false,
      "listChanged": true
    },
    "tools": {
      "listChanged": true
    },
    "extensions": {
      "io.modelcontextprotocol/ui": {}
    }
  },
  "serverInfo": {
    "name": "demo",
    "version": " ...
   (협상된 세션 ID: 484a9f499d79491388fd2bb279917b9c)

2) notifications/initialized 전송 완료

3) tools/list 응답
   - add: 두 정수를 더합니다.
   - multiply: 두 정수를 곱합니다.
   - get_greeting: 인사말을 생성합니다.

4) tools/call 응답
{
  "_meta": {
    "fastmcp": {
      "wrap_result": true
    }
  },
  "content": [
    {
      "type": "text",
      "text": "3 + 7 = 10"
    }
  ],
  "structuredContent": {
    "result": "3 + 7 = 10"
  },
  "isError": false
}


In [9]:
# tools 외에 resources와 prompts도 같은 세션(session_headers)에서 조회합니다.
def rpc(method, params=None, _id=10):
    """앞에서 협상한 세션으로 JSON-RPC를 호출하고 result를 돌려줍니다."""
    with httpx.Client(timeout=10.0) as cli:
        r = cli.post(URL, headers=session_headers, json={
            'jsonrpc': '2.0', 'id': _id, 'method': method, 'params': params or {},
        })
    return _parse_sse(r.text)['result']

# 1) Resource: 모델이 호출하는 게 아니라 URI로 읽어오는 데이터
print('resources/list')
for r in rpc('resources/list')['resources']:
    print(f"   - {r['uri']}: {r.get('description', '')[:40]}")

print('\nresources/templates/list  (URI에 인자를 넣는 동적 Resource)')
for r in rpc('resources/templates/list')['resourceTemplates']:
    print(f"   - {r['uriTemplate']}: {r.get('description', '')[:40]}")

print('\nresources/read')
print('   config://limits ->', rpc('resources/read', {'uri': 'config://limits'})['contents'][0]['text'])
print('   const://pi      ->', rpc('resources/read', {'uri': 'const://pi'})['contents'][0]['text'])

# 2) Prompt: 등록한 템플릿에 인자를 채워 완성된 메시지를 받습니다.
print('\nprompts/list')
for p in rpc('prompts/list')['prompts']:
    arg_names = ', '.join(a['name'] for a in p.get('arguments', []))
    print(f"   - {p['name']}({arg_names}): {p.get('description', '')[:40]}")

print('\nprompts/get explain_calculation(expression="3 + 7 * 5")')
got = rpc('prompts/get', {'name': 'explain_calculation',
                          'arguments': {'expression': '3 + 7 * 5'}})
print('   ->', got['messages'][0]['content']['text'][:60], '...')

resources/list
   - info://about: 서버 정보를 반환합니다.
   - config://limits: 이 서버의 계산 정책(지원 연산, 입력 한도)을 반환합니다.

resources/templates/list  (URI에 인자를 넣는 동적 Resource)
   - const://{name}: 이름으로 상수 값을 조회합니다. (pi, e, golden)

resources/read
   config://limits -> {"supported_ops": ["add", "multiply"], "max_operand": 10000000, "note": "\uc815\uc218 \uc5f0\uc0b0\ub9cc \uc9c0\uc6d0\ud569\ub2c8\ub2e4."}
   const://pi      -> 3.141592653589793

prompts/list
   - explain_calculation(expression): 주어진 수식을 단계별로 풀이해 설명하도록 지시하는 프롬프트.

prompts/get explain_calculation(expression="3 + 7 * 5")
   -> 다음 수식을 초등학생도 이해할 수 있게 한 단계씩 풀이해 주세요.

수식: 3 + 7 * 5

각 단계에서  ...


In [10]:
# Prompt의 가치: 서버가 제공한 프롬프트 메시지를 그대로 모델에 넣어 실행합니다.
got = rpc('prompts/get', {'name': 'explain_calculation',
                          'arguments': {'expression': '128 / 4 + 6'}})
prompt_text = got['messages'][0]['content']['text']

print('서버가 제공한 프롬프트:')
print(prompt_text)

print('\n--- 이 프롬프트로 모델 호출 ---')
resp = await model.ainvoke(prompt_text)
print(resp.text)

서버가 제공한 프롬프트:
다음 수식을 초등학생도 이해할 수 있게 한 단계씩 풀이해 주세요.

수식: 128 / 4 + 6

각 단계에서 무엇을 왜 계산하는지 한 줄로 설명하고, 마지막 줄에 최종 답을 적어 주세요.

--- 이 프롬프트로 모델 호출 ---
1) **나누기를 먼저 계산해요**: 128을 4로 나누면 몇 묶음이 되는지 알아보는 거예요.  
   → 128 ÷ 4 = 32

2) **그다음 더하기를 해요**: 방금 나온 32에 6을 더해서 모두 합친 개수를 구해요.  
   → 32 + 6 = 38

**최종 답: 38**


### Tool vs Resource vs Prompt

| 구성 요소 | 메서드 | 호출 주체 |
| --- | --- | --- |
| Tool | `tools/call` | 모델이 판단해 자동 호출 |
| Resource | `resources/read` | 클라이언트가 필요할 때 읽음 |
| Prompt | `prompts/get` | 사용자가 가져와 모델에 넣음 |

Resource/Prompt는 LLM이 호출해야 하는 Tool과 다르게 연결 과정에서 바로 가져올 수 있는 요소들입니다.

LangChain에서는 위 통신 과정을 MultiServerMCPClient로 추상화하여 지원합니다.

In [14]:
from langchain_mcp_adapters.client import MultiServerMCPClient

# MCP 서버 연결 설정
client = MultiServerMCPClient({
    'demo': {
        'transport': 'http',
        'url': 'http://localhost:8090/mcp',
    }
})

# 도구 로드
tools = await client.get_tools()
print(f'로드된 도구 수: {len(tools)}')
for t in tools:
    print(f'  - {t.name}: {t.description[:50]}...')

# Resource와 Prompt는 모델이 자동 호출하는 대상이 아니라, 이 목록에는 Tool만 잡힙니다.
print('(config://limits, const://{name}, explain_calculation은 자동 도구가 아니므로 빠져 있습니다)')

로드된 도구 수: 3
  - add: 두 정수를 더합니다....
  - multiply: 두 정수를 곱합니다....
  - get_greeting: 인사말을 생성합니다....
(config://limits, const://{name}, explain_calculation은 자동 도구가 아니므로 빠져 있습니다)


### MCP 도구를 Agent에 연결

In [15]:
from langchain.agents import create_agent
from langgraph.checkpoint.memory import InMemorySaver
import uuid

agent = create_agent(
    model,
    tools=tools,
    system_prompt='도구를 사용할 때마다, 중간 과정을 간략히 설명하세요.',
    checkpointer=InMemorySaver()
)
print(f'Agent 생성 완료')

Agent 생성 완료


In [17]:
thread_id = str(uuid.uuid4())
# 랜덤 문자열 (매번 달라지게 하기 위해)

config = {'configurable': {'thread_id': thread_id}}

result = await agent.ainvoke(
    {'messages': [{'role': 'user', 'content': '3과 7을 더하고, 그 결과에 5를 곱해줘.'}]},
    config = config)

print(result['messages'][-1].text)

3 + 7 = 10, 그리고 10 × 5 = **50** 입니다.


In [18]:
# 인사말 도구 사용
result = await agent.ainvoke({
    'messages': [{'role': 'user', 'content': "'철수'에게 캐주얼한 인사말을 만들어줘."}]},
    config = config)
print(result['messages'][-1].text)

안녕 철수! 반가워~


In [19]:
# 메시지 흐름 확인
for i, msg in enumerate(result['messages']):
    msg_type = type(msg).__name__
    if hasattr(msg, 'tool_calls') and msg.tool_calls:
        calls = ', '.join([f"{tc['name']}({tc['args']})" for tc in msg.tool_calls])
        print(f'[{i}] {msg_type}: tool_calls=[{calls}]')
    elif hasattr(msg, 'tool_call_id'):
        print(f'[{i}] {msg_type} ({msg.name}): {str(msg.text)[:80]}')
    else:
        print(f'[{i}] {msg_type}: {msg.text[:80]}')

[0] HumanMessage: 3과 7을 더하고, 그 결과에 5를 곱해줘.
[1] AIMessage: tool_calls=[add({'a': 3, 'b': 7})]
[2] ToolMessage (add): 3 + 7 = 10
[3] AIMessage: tool_calls=[multiply({'a': 10, 'b': 5})]
[4] ToolMessage (multiply): 10 x 5 = 50
[5] AIMessage: 3 + 7 = 10, 그리고 10 × 5 = **50** 입니다.
[6] HumanMessage: '철수'에게 캐주얼한 인사말을 만들어줘.
[7] AIMessage: tool_calls=[get_greeting({'name': '철수', 'style': 'casual'})]
[8] ToolMessage (get_greeting): 안녕 철수! 반가워~
[9] AIMessage: 안녕 철수! 반가워~


---
## build_agent로 에이전트 스크립트 저장하기

이전에 구성한 select_model과 MCP 서버를 추가하여, 에이전트 스크립트를 개선합니다.

In [20]:
%%writefile build_agent.py
"""build_agent.py

에이전트를 한 곳에서 조립하는 팩토리입니다. 모델은 select_model로 고르고,
표준 도구(tools.py)에 MCP 서버 도구를 더해 에이전트를 만듭니다. 

MCP 도구는 await로 처리해야 하므로 build_agent는 async 함수로 정의합니다. 
"""

from __future__ import annotations

from langchain.agents import create_agent
from langchain_mcp_adapters.client import MultiServerMCPClient

from select_model import load_model
from tools import DEFAULT_TOOLS

DEFAULT_SYSTEM_PROMPT = (
    '당신은 유용한 AI 어시스턴트입니다. '
    '필요하면 도구를 사용하고, 도구를 실행하기 전 중간 과정을 간략히 설명하세요.'
)

# build_agent가 직접 연결하는 MCP 서버. 서버가 실행 중이어야 도구가 로드됩니다.
MCP_SERVERS = {
    'demo': {'transport': 'streamable_http', 'url': 'http://localhost:8090/mcp'},
    
}


async def load_mcp_tools(servers=None):
    """MCP 서버에서 도구를 받아옵니다. 연결에 실패하면 빈 목록을 돌려줍니다."""
    servers = MCP_SERVERS if servers is None else servers
    if not servers:
        return []
    try:
        client = MultiServerMCPClient(servers)
        return await client.get_tools()
    except Exception as e:
        print(f'[build_agent] MCP 도구 로드 실패, 표준 도구만 사용합니다: {e}')
        return []


async def build_agent(
    model_name=None,
    platform='openai',
    model=None,
    tools=None,
    system_prompt=None,
    checkpointer=None,
    mcp_servers=None,
    **model_kwargs,
):
    """현재까지 구성된 에이전트를 만들어 반환합니다.

    Args:
        model_name: 모델 이름. select_model.load_model로 전달됩니다.
        platform: 'openai' | 'vllm' | 'ollama'. load_model로 전달됩니다.
        model: 이미 만든 chat model. 주면 model_name/platform을 무시합니다.
        tools: 도구 목록. 생략 시 tools.py의 DEFAULT_TOOLS을 사용합니다.
        system_prompt: 시스템 프롬프트. 생략 시 기본값을 씁니다.
        checkpointer: 멀티턴 대화를 위한 체크포인터.
        mcp_servers: 연결할 MCP 서버 설정. 기본값은 MCP_SERVERS, {}면 MCP를 사용하지 않습니다.
        **model_kwargs: load_model로 전달되는 추가 인자 (reasoning_effort 등).
    """
    if model is None:
        model = load_model(model_name, platform=platform, **model_kwargs)
    base_tools = list(DEFAULT_TOOLS) if tools is None else list(tools)
    mcp_tools = await load_mcp_tools(mcp_servers)
    if system_prompt is None:
        system_prompt = DEFAULT_SYSTEM_PROMPT
    return create_agent(
        model,
        tools=base_tools + mcp_tools,
        system_prompt=system_prompt,
        checkpointer=checkpointer,
    )


Overwriting build_agent.py


데모 MCP 서버가 구동 중인 상태에서, 아래의 코드로 에이전트를 실행해 보겠습니다.

In [21]:
from select_model import load_model
from build_agent import build_agent

PLATFORM = 'openai'        # 'openai' | 'vllm' | 'ollama'

MODEL_NAME = {
    'openai': 'gpt-5.2',
    'vllm': None,
    'ollama': 'qwen3.6:latest',
}[PLATFORM]

MODEL_KWARGS = {
    'openai': {'reasoning_effort': 'low'},
    'vllm': {},
    'ollama': {},
}[PLATFORM]

model = load_model(MODEL_NAME, platform=PLATFORM, **MODEL_KWARGS)
print(f'{PLATFORM} 모델 로드:', type(model).__name__)

agent = await build_agent(model_name=MODEL_NAME, platform=PLATFORM, **MODEL_KWARGS)

result = await agent.ainvoke({
    'messages': [{'role': 'user', 'content': "'영희'에게 캐주얼한 인사말을 만들고, 2의 8승도 계산해줘."}]
})
print(result['messages'][-1].text)

openai 모델 로드: ChatOpenAI
- 캐주얼 인사말: **안녕 영희! 반가워~**
- 2의 8승: **256**


In [22]:
from stream_utils import stream_with_markdown
await stream_with_markdown(agent, "'영희'에게 캐주얼한 인사말을 만들고, 2의 8승도 계산해줘.")

인사말 생성과 거듭제곱 계산을 각각 도구로 처리하겠습니다.

🔧 Tool 호출: `get_greeting` (인자: `{'name': '영희', 'style': 'casual'}`)

🔧 Tool 호출: `calculator` (인자: `{'expression': '2 ** 8'}`)

✅ 결과: 256

✅ 결과: 안녕 영희! 반가워~


---
- 인사말(캐주얼): **안녕 영희! 반가워~**
- 2의 8승: **256**

'인사말 생성과 거듭제곱 계산을 각각 도구로 처리하겠습니다.- 인사말(캐주얼): **안녕 영희! 반가워~**\n- 2의 8승: **256**'

## 4. STDIO MCP 서버 연결하기


STDIO MCP 서버는 서버를 미리 켜 놓는 대신, 에이전트의 실행 과정에서 직접 커맨드로 실행하는 방식입니다.

In [24]:
%%writefile weather_server.py
from typing import Any
import httpx
from mcp.server.fastmcp import FastMCP

# FastMCP 서버 초기화
mcp = FastMCP("weather")

# 상수
NWS_API_BASE = "https://api.weather.gov"
USER_AGENT = "weather-app/1.0"

# ⚠️ 참고: 아래 verify 인자는 다양한 환경에서의 SSL 인증서 문제를
# 우회하기 위한 학습용 옵션입니다. production에서는 반드시 verify=True (또는 인증서 번들 경로)로 두고,
# 실패하면 환경의 CA 인증서를 점검하세요.
async def make_nws_request(url: str, verify: bool) -> dict[str, Any] | None:
    """NWS API를 통한 기상 데이터 요청 (verify=False는 학습용)"""
    headers = {
        "User-Agent": USER_AGENT,
        "Accept": "application/geo+json"
    }
    async with httpx.AsyncClient(verify=verify) as client:
        try:
            response = await client.get(url, headers=headers, timeout=30.0)
            response.raise_for_status()
            return response.json()
        except Exception:
            return None

def format_alert(feature: dict) -> str:
    """기상경보 정보를 포맷팅하여 문자열로 반환환"""
    props = feature["properties"]
    return f"""
Event: {props.get('event', 'Unknown')}
Area: {props.get('areaDesc', 'Unknown')}
Severity: {props.get('severity', 'Unknown')}
Description: {props.get('description', 'No description available')}
Instructions: {props.get('instruction', 'No specific instructions provided')}
"""


@mcp.tool()
async def get_alerts(state: str) -> str:
    """미국 주의 기상 경보 수집

    Args:
        state: 2글자로 구성된 미국 주 코드 (예: CA, NY)
    """
    url = f"{NWS_API_BASE}/alerts/active/area/{state}"
    data = await make_nws_request(url, verify=False)

    if not data or "features" not in data:
        return "Unable to fetch alerts or no alerts found."

    if not data["features"]:
        return "No active alerts for this state."

    alerts = [format_alert(feature) for feature in data["features"]]
    return "\n---\n".join(alerts)

@mcp.tool()
async def get_forecast(latitude: float, longitude: float) -> str:
    """특정 위-경도 위치의 일기예보 수집

    Args:
        latitude: 위치의 위도
        longitude: 위치의 경도
    """
    # 예보 그리드 엔드포인트 호출
    points_url = f"{NWS_API_BASE}/points/{latitude},{longitude}"
    points_data = await make_nws_request(points_url, verify=False)

    if not points_data:
        return "Unable to fetch forecast data for this location."

    # points 응답에서 예보 URL 추출
    forecast_url = points_data["properties"]["forecast"]
    forecast_data = await make_nws_request(forecast_url, verify=False)

    if not forecast_data:
        return "Unable to fetch detailed forecast."

    # 기간들을 읽기 쉬운 예보로 포맷팅
    periods = forecast_data["properties"]["periods"]
    forecasts = []
    for period in periods[:5]:  # 다음 5개 기간만 출력
        forecast = f"""
{period['name']}:
Temperature: {period['temperature']}°{period['temperatureUnit']}
Wind: {period['windSpeed']} {period['windDirection']}
Forecast: {period['detailedForecast']}
"""
        forecasts.append(forecast)

    return "\n---\n".join(forecasts)

if __name__ == "__main__":
    # 서버 초기화 및 실행
    print("Starting weather server...")
    mcp.run(transport='stdio')


Overwriting weather_server.py


Windows/Jupyter Notebook 조합에서는 STDIO 실행이 불가능하므로, 스크립트를 통해 실행합니다.

In [30]:
%%writefile stdio_mcp_example.py
"""
MCP STDIO 클라이언트 (langchain_mcp_adapters 기반)

weather_server.py에 STDIO 방식으로 연결하여
에이전트를 구성하고 도구를 호출합니다.
"""

import asyncio
import sys
import os

from dotenv import load_dotenv

load_dotenv(".env", override=True)

from langchain_mcp_adapters.client import MultiServerMCPClient
from langchain.agents import create_agent
from select_model import load_model

PLATFORM = "openai"

# .venv의 python 경로 (Windows에서는 Scripts\\python.exe, macOS/Linux에서는 bin/python)
BASE_DIR = os.path.dirname(os.path.abspath(__file__))
if sys.platform == "win32":
    PYTHON = os.path.join(BASE_DIR, ".venv", "Scripts", "python.exe")
else:
    PYTHON = os.path.join(BASE_DIR, ".venv", "bin", "python")

async def main():
    # STDIO 방식으로 MCP 서버 연결
    client = MultiServerMCPClient(
        {
            "weather": {
                "command": PYTHON,
                "args": [os.path.join(BASE_DIR, "weather_server.py")],
                "transport": "stdio",
            },
        }
    )

    # MCP 도구 로드
    tools = await client.get_tools()
    print(f"로드된 도구 수: {len(tools)}")
    for t in tools:
        print(f"  - {t.name}: {t.description}")

    # 에이전트 생성
    model = load_model("gpt-5.2" if PLATFORM == "openai" else None, platform=PLATFORM)
    agent = create_agent(
        model,
        tools=tools,
        system_prompt="중간 과정을 매번 간략히 설명하세요.",
    )


    # Weather 도구 테스트
    print("\n--- Weather 도구 테스트 ---")
    result = await agent.ainvoke(
        {"messages": [{"role": "user", "content": "캘리포니아(CA)의 현재 기상 경보를 확인해줘."}]}
    )
    print(result["messages"][-1].text)


if __name__ == "__main__":
    asyncio.run(main())

Overwriting stdio_mcp_example.py


### [참고] stdio --> streamable-http   
STDIO 서버를 streamble-http로 우회해 연결하고 싶은 경우, `supergateway`를 이용할 수 있습니다.

In [ ]:
# node.js 설치 필수
# 아래 명령을 프로젝트 폴더에서 별도 터미널로 실행하면, stdio MCP 서버를 HTTP 포트로 노출합니다.

# npx -y supergateway --stdio "C:\lab\.venv\Scripts\python.exe C:\lab\weather_server.py" --outputTransport streamableHttp  --port 8000

In [ ]:
# 파일시스템 MCP
# npx -y supergateway --stdio "npx -y @modelcontextprotocol/server-filesystem C:/lab" --port 8010 --outputTransport streamableHttp

In [ ]:
# supergateway가 노출한 HTTP 엔드포인트로 연결
client = MultiServerMCPClient(
    {
        "weather": {
            "transport": "streamable_http",
            "url": "http://localhost:8000/mcp/",
        },
        "filesystem":{
            "transport": "streamable_http",
            "url": "http://localhost:8010/mcp/"
        }
    }
)

tools = await client.get_tools()
print(f'로드된 도구 수: {len(tools)}')
for t in tools:
    print(f'  - {t.name}: {t.description[:50]}...')

In [ ]:
from stream_utils import stream_with_markdown

In [ ]:
agent = create_agent(
    model,
    tools=tools,
    system_prompt="중간 과정을 매번 간략히 설명하세요.",
    checkpointer= InMemorySaver()
)


thread_id = str(uuid.uuid4())
config = {'configurable': {'thread_id': thread_id}}

# # Weather 도구 테스트
print("\n--- Weather 도구 테스트 ---")
# result = await agent.ainvoke(
#     {"messages": [{"role": "user", "content": "캘리포니아(CA)의 현재 기상 경보를 확인해줘."}]},
#     config = config
# )
# print(result["messages"][-1].text)

await stream_with_markdown(agent, "캘리포니아(CA)의 현재 기상 경보를 확인해줘.",
                     config = config)

In [ ]:
# 파일시스템 도구 테스트
await stream_with_markdown(agent, "지금 내용을 마크다운 보고서로 폴더에 저장해줘.",
                     config = config)

---
## Agent의 Context 누적

도구가 늘어날수록 기능이 확장되지만, 도구 스키마와 호출 결과를 저장해야 하므로 컨텍스트가 증가하게 됩니다.

In [ ]:
from langchain.messages import AIMessage, ToolMessage
import json

def schema_chars(t):
    sch = getattr(t, 'args_schema', None)
    js = sch.model_json_schema() if hasattr(sch, 'model_json_schema') else {}
    return len(t.name) + len(t.description or '') + len(json.dumps(js, ensure_ascii=False))

overhead = sum(schema_chars(t) for t in tools)
print(f'도구 {len(tools)}개의 스키마 분량: 약 {overhead:,}자')

# 도구 호출이 이어지면서 입력 컨텍스트가 증가
result = await agent.ainvoke(
    {'messages': [{'role': 'user', 'content': '캘리포니아(CA)의 현재 기상 경보를 확인해줘.'}]},
    config={'configurable': {'thread_id': 'mcp-token-cost'}},
)
for i, m in enumerate([m for m in result['messages'] if isinstance(m, AIMessage) and m.usage_metadata], 1):
    u = m.usage_metadata
    print(f'LLM 호출 {i}: 입력 {u["input_tokens"]:>6} 토큰, 출력 {u["output_tokens"]:>5} 토큰')

tool_msgs = [m for m in result['messages'] if isinstance(m, ToolMessage)]
print(f'도구 결과 {len(tool_msgs)}개, 총 {sum(len(str(m.content)) for m in tool_msgs):,}자')

---
## 정리

### 이번 실습에서 배운 것

- MCP: LLM에 도구와 데이터를 제공하기 위한 표준 통신 규약
- FastMCP: `@mcp.tool()`, `@mcp.resource()`, `@mcp.prompt()`로 간단하게 MCP 서버 구축
- Tool vs Resource vs Prompt: 모델이 자동 호출하는 건 Tool뿐이고, Resource는 클라이언트가 읽고 Prompt는 사용자가 가져와 모델에 넣음
- Streamable-HTTP 트랜스포트: 웹 기반으로 `http://host:port/mcp`로 접속
- STDIO 트랜스포트: STDIO 기반으로 커맨드를 통해 실행 (윈도우 + 주피터 환경에서는 작동 어려움)
- JSON-RPC 메시지 흐름: `initialize` 후 `tools/list`와 `tools/call`을 한 번 직접 확인
- MultiServerMCPClient: 여러 MCP 서버의 도구를 한 번에 LangChain Agent로 로드
- Context 누적: 도구 스키마와 호출 결과가 매 요청에 실려, 도구가 늘수록 입력 토큰이 커짐
